# ML HOL - PART 2
## ML Feature Store

- In this notebook, we will walk through a few transformations that profile and cleanse raw data.
- We will then store these transformations into a Feature Store that will automatically write out a transformed table on a daily basis



To get started, let's select a few packages that we will need. In the **Packages** drop-down picker in the top right of the UI, search for and add the following packages by clicking on them:

- snowflake-ml-python

Once you add the packages, click the **Start** button! Once it says **Active**, you're ready to run the rest of the Notebook.

### Import Libraries

In [ ]:
# Snowpark for Python
from snowflake.snowpark.types import DoubleType
import snowflake.snowpark.functions as F

### Setup and establish Secure Connection to Snowflake

Notebooks establish a Snowpark Session when the notebook is attached to the kernel. We will use an assigned warehouse, database, and schema that will be used throughout this tutorial.

In [ ]:
# Get Snowflake Session object
session = get_active_session()
session.sql_simplifier_enabled = True

# Add a query tag to the session.
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"e2e_ml_snowparkpython", 
                     "version":{"major":1, "minor":0,},
                     "attributes":{"is_hol":1, "source":"notebook"}}

# Current Environment Details
print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

### 1. Read in the raw `diamonds` data



In [ ]:
# Load in the data
raw_diamonds_df = session.table("RAW_DIAMONDS")
raw_diamonds_df

### 2. Data Profiling

In [ ]:
# Look at descriptive stats on the DataFrame
raw_diamonds_df.describe()

### 3. Data cleansing

Let's start creating a cleansed data frame and force headers to uppercase using Snowpark DataFrame operations for standardization when columns are later written to a Snowflake table.

***Note: Snowpark data frames go through lazy execution which can reduce data transfer time/costs: https://docs.snowflake.com/en/developer-guide/snowpark/index#reduced-data-transfer***

In [ ]:
raw_diamonds_df.columns

In [ ]:
# create a new dataframe for cleansing
diamonds_df = raw_diamonds_df

# Force headers to uppercase
for colname in diamonds_df.columns:
    if colname == '"table"':
       new_colname = "TABLE_PCT"
    else:
        new_colname = str.upper(colname)
    diamonds_df = diamonds_df.with_column_renamed(colname, new_colname)

diamonds_df

In [ ]:
diamonds_df.columns

Next, we standardize the category formatting for `CUT` using Snowpark DataFrame operations and regex.

This way, when we write to a Snowflake table, there will be no inconsistencies in how the Snowpark DataFrame will read in the category names. Secondly, the feature transformations on categoricals will be easier to encode.

In [ ]:
diamonds_df.select(F.col('CUT')).distinct().show()

In [ ]:
def fix_values(columnn):
    return F.upper(F.regexp_replace(F.col(columnn), '[^a-zA-Z0-9]+', '_'))

for col in ["CUT"]:
    diamonds_df = diamonds_df.with_column(col, fix_values(col))

diamonds_df.select(F.col('CUT')).distinct().show()

Finally, let's check the schema and cast any decimal types to DoubleType()

***Note: DecimalType() isn't support by Snowpark ML at the moment.***

In [ ]:
list(diamonds_df.schema)

In [ ]:
for colname in ["CARAT", "X", "Y", "Z", "DEPTH", "TABLE_PCT"]:
    diamonds_df = diamonds_df.with_column(colname, diamonds_df[colname].cast(DoubleType()))

list(diamonds_df.schema)

## Put Feature Transformations into a Feature Store

Here is the back-end data model

### Feature store object    | Snowflake object
- feature store             | schema
- feature entity            | tag
- feature view              | dynamic table or view
- feature                   | column in a dynamic table or in a view

### Create the Feature Store
https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/feature_store/snowflake.ml.feature_store.FeatureStore

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

fs = FeatureStore(
        session=session,
        database="ML_HOL_DB",
        name=session.get_current_schema(),  
        default_warehouse=session.get_current_warehouse(), 
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
     )

### Create A Feature Entity
https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/feature_store/snowflake.ml.feature_store.Entity

In [ ]:
from snowflake.ml.feature_store import Entity

eDiamond = Entity(
    name="DIAMONDS",
    join_keys=["DIAMONDS_ID"],
    desc="Diamond Identifier"
)
fs.register_entity(eDiamond)

fs.list_entities().show()

### Create A Feature View

This will store all the feature code and set a schedule for refresh

https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/feature_store/snowflake.ml.feature_store.FeatureView

In [ ]:
from snowflake.ml.feature_store import FeatureView

# fv is a local object that hasn't materiaized to Snowflake backend yet.
fv = FeatureView(
    name="DIAMONDS",
    entities=[eDiamond],
    feature_df=diamonds_df ,                # Snowpark DataFrame containing feature transformations
    timestamp_col=None,                     # optional timestamp column name in the dataframe
    refresh_freq='1 day',                   # how often feature data refreshes (NULL = VIEW)
    desc="Cleansed Diamond data"            # optional description
)

# fv is a local object that maps to a Snowflake backend object (dynamic table in this case)
fv1 = fs.register_feature_view(fv, '1.0')

fs.list_feature_views().select('NAME', 'VERSION', 'DESC', 'REFRESH_FREQ').show()

In [ ]:
diamonds_df.explain()

### Update feature views

After a feature view is registered, it is materialized to Snowflake backend. You can still update some metadata for a registered feature view with `update_feature_view`. Below cell updates the `desc` of a managed feature view. You can check our [API reference](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/feature_store/snowflake.ml.feature_store.FeatureStore) page to find the full list of metadata that can be updated.

In [ ]:
# This shows how to pull all feature views and interate through (even though we have just 1 fv right now)
for row in fs.list_feature_views().collect():
    fv = fs.get_feature_view(row['NAME'], row['VERSION'])
    update_fv = fs.update_feature_view(row['NAME'], row['VERSION'], desc=f'Updated: {fv.desc}')

fs.list_feature_views().select('NAME', 'VERSION', 'DESC').show()

### Operate feature views

For **managed feature views**, you can suspend, resume, or manually refresh the backend pipelines. A managed feature view is an automated feature pipeline that computes the features on a given schedule. You create a managed feature view by setting the `refresh_freq`. In contrast, a **static feature view** is created when `refresh_freq` is set to None.

In [ ]:
registered_fv = fs.get_feature_view('DIAMONDS', '1.0')
fs.list_feature_views().select('name', 'version', 'desc', 'refresh_freq', 'scheduling_state').show()

In [ ]:
suspended_fv = fs.suspend_feature_view(registered_fv)
fs.list_feature_views().select('name', 'version', 'desc', 'refresh_freq', 'scheduling_state').show()

In [ ]:
# refresh with name and version
fs.get_refresh_history('DIAMONDS', '1.0').show()

In [ ]:
#Create link to feature store UI to inspect newly created feature view!
import streamlit as st

org_name = session.sql('SELECT CURRENT_ORGANIZATION_NAME()').collect()[0][0]
account_name = session.sql('SELECT CURRENT_ACCOUNT_NAME()').collect()[0][0]
db_name = session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]
schema_name = session.sql('SELECT CURRENT_SCHEMA()').collect()[0][0]

st.write(f'https://app.snowflake.com/{org_name}/{account_name}/#/features/database/{db_name}/store/{schema_name}')

In the next notebook, we will perform data transformations with the Snowpark ML Preprocessing API for feature engineering. 

Before proceeding, let's review our new new object.

- In Snowsight left nav bar, drill into Data » Databases.

- Select a database, schema, and diamonds table.

- Select the Lineage tab.